To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
!pip install qwen-tts
!pip install datasets soundfile librosa
!git clone https://github.com/QwenLM/Qwen3-TTS.git

### Qwen3-TTS

[Qwen3-TTS](https://github.com/QwenLM/Qwen3-TTS) is an open-source TTS model series from Alibaba's Qwen team supporting voice cloning, voice design, and multilingual speech generation across 10 languages. We use the **1.7B Base** model here, which is the recommended size for fine-tuning.

The fine-tuning pipeline uses the official `qwen-tts` package to:
1. Encode audio into discrete codec tokens using the Qwen3-TTS-Tokenizer-12Hz
2. Train the talker model via supervised fine-tuning (SFT)

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel
from huggingface_hub import snapshot_download

MODEL_PATH = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

# Download model files locally (needed for checkpoint saving later)
local_model_path = snapshot_download(MODEL_PATH)

qwen3tts = Qwen3TTSModel.from_pretrained(
    MODEL_PATH,
    device_map = "cuda:0",
    dtype = torch.bfloat16,
)
print(f"Model loaded: {MODEL_PATH}")
print(f"Model parameters: {sum(p.numel() for p in qwen3tts.model.parameters()) / 1e6:.1f}M")

<a name="Data"></a>
### Data Prep

We use the `MrDragonFox/Elise` dataset, which is designed for TTS model training. The pipeline:
1. Export audio samples to `.wav` files
2. Create a JSONL file with `audio`, `text`, and `ref_audio` fields
3. Encode audio into codec tokens using `Qwen3-TTS-Tokenizer-12Hz`

You can replace this with your own dataset — just ensure each sample has a `.wav` audio file and its text transcript.

In [ ]:
#@title Export Dataset to WAV Files

import os
import json
import soundfile as sf
import numpy as np
from datasets import load_dataset

dataset = load_dataset("MrDragonFox/Elise", split = "train")

# Create directories for audio files
os.makedirs("data/train_audio", exist_ok = True)

# Use a subset for demo (adjust for full training)
NUM_SAMPLES = 50  # Increase for better results
dataset = dataset.select(range(min(NUM_SAMPLES, len(dataset))))

# Export audio to wav and create JSONL
TARGET_SR = 24000  # Qwen3-TTS expects 24kHz audio
raw_jsonl_path = "data/train_raw.jsonl"

# Use the first sample as reference audio for all samples
# (recommended by official docs for better speaker consistency)
ref_audio = dataset[0]["audio"]
ref_wav = np.array(ref_audio["array"], dtype = np.float32)
ref_sr = ref_audio["sampling_rate"]

if ref_sr != TARGET_SR:
    import librosa
    ref_wav = librosa.resample(ref_wav, orig_sr = ref_sr, target_sr = TARGET_SR)

ref_audio_path = "data/ref_speaker.wav"
sf.write(ref_audio_path, ref_wav, TARGET_SR)

with open(raw_jsonl_path, "w") as f:
    for i, sample in enumerate(dataset):
        audio = sample["audio"]
        wav = np.array(audio["array"], dtype = np.float32)
        sr = audio["sampling_rate"]

        # Resample to 24kHz if needed
        if sr != TARGET_SR:
            import librosa
            wav = librosa.resample(wav, orig_sr = sr, target_sr = TARGET_SR)

        # Save wav file
        wav_path = f"data/train_audio/utt_{i:04d}.wav"
        sf.write(wav_path, wav, TARGET_SR)

        # Get text transcript
        text = sample.get("text", sample.get("transcription", f"Sample {i}"))

        # Write JSONL entry
        entry = {
            "audio": wav_path,
            "text": text,
            "ref_audio": ref_audio_path,
        }
        f.write(json.dumps(entry, ensure_ascii = False) + "\n")

print(f"Exported {len(dataset)} samples to {raw_jsonl_path}")
print(f"Reference audio saved to {ref_audio_path}")

In [ ]:
#@title Encode Audio to Codec Tokens

import json
from qwen_tts import Qwen3TTSTokenizer

TOKENIZER_MODEL = "Qwen/Qwen3-TTS-Tokenizer-12Hz"
INPUT_JSONL = "data/train_raw.jsonl"
OUTPUT_JSONL = "data/train_with_codes.jsonl"
BATCH_SIZE = 8  # Reduce if running out of memory

# Load tokenizer
tokenizer_12hz = Qwen3TTSTokenizer.from_pretrained(
    TOKENIZER_MODEL,
    device_map = "cuda:0",
)

# Read input data
with open(INPUT_JSONL) as f:
    total_lines = [json.loads(line.strip()) for line in f]

# Encode in batches
final_lines = []
for start in range(0, len(total_lines), BATCH_SIZE):
    batch = total_lines[start:start + BATCH_SIZE]
    batch_audios = [line["audio"] for line in batch]

    enc_res = tokenizer_12hz.encode(batch_audios)
    for code, line in zip(enc_res.audio_codes, batch):
        line["audio_codes"] = code.cpu().tolist()
        final_lines.append(line)

    print(f"  Encoded {min(start + BATCH_SIZE, len(total_lines))}/{len(total_lines)} samples")

# Write output
with open(OUTPUT_JSONL, "w") as f:
    for line in final_lines:
        f.write(json.dumps(line, ensure_ascii = False) + "\n")

print(f"\nTokenized data saved to {OUTPUT_JSONL}")
print(f"Total samples: {len(final_lines)}")

# Free tokenizer memory
del tokenizer_12hz
torch.cuda.empty_cache()

<a name="Train"></a>
### Train the model
Now let's fine-tune Qwen3-TTS! We use the official training pipeline from the [Qwen3-TTS repo](https://github.com/QwenLM/Qwen3-TTS/tree/main/finetuning). We do a short run (3 epochs) to demonstrate; increase `NUM_EPOCHS` for better results.

**Note:** The training uses bfloat16 mixed precision for numerical stability.

In [ ]:
#@title Run Fine-tuning

import os
import sys
import json
import shutil

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoConfig

# Add the finetuning scripts to path
sys.path.insert(0, "Qwen3-TTS/finetuning")
from dataset import TTSDataset

TRAIN_JSONL = "data/train_with_codes.jsonl"
OUTPUT_DIR = "output"
SPEAKER_NAME = "elise"
BATCH_SIZE = 1
LR = 2e-5
NUM_EPOCHS = 3

# Load training data
train_data = [json.loads(line) for line in open(TRAIN_JSONL)]
config = AutoConfig.from_pretrained(MODEL_PATH)
dataset = TTSDataset(train_data, qwen3tts.processor, config)
train_dataloader = DataLoader(
    dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = dataset.collate_fn,
)

# Setup optimizer
optimizer = AdamW(qwen3tts.model.parameters(), lr = LR, weight_decay = 0.01)

# Training loop
qwen3tts.model.train()
target_speaker_embedding = None

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    num_steps = 0

    for step, batch in enumerate(train_dataloader):
        input_ids = batch["input_ids"].to("cuda")
        codec_ids = batch["codec_ids"].to("cuda")
        ref_mels = batch["ref_mels"].to("cuda")
        text_embedding_mask = batch["text_embedding_mask"].to("cuda")
        codec_embedding_mask = batch["codec_embedding_mask"].to("cuda")
        attention_mask = batch["attention_mask"].to("cuda")
        codec_0_labels = batch["codec_0_labels"].to("cuda")
        codec_mask = batch["codec_mask"].to("cuda")

        model = qwen3tts.model

        # Extract speaker embedding from reference audio
        speaker_embedding = model.speaker_encoder(
            ref_mels.to(model.dtype)
        ).detach()
        if target_speaker_embedding is None:
            target_speaker_embedding = speaker_embedding

        # Build dual-channel input embeddings
        input_text_ids = input_ids[:, :, 0]
        input_codec_ids = input_ids[:, :, 1]

        input_text_embedding = model.talker.model.text_embedding(input_text_ids) * text_embedding_mask
        input_codec_embedding = model.talker.model.codec_embedding(input_codec_ids) * codec_embedding_mask
        input_codec_embedding[:, 6, :] = speaker_embedding

        input_embeddings = input_text_embedding + input_codec_embedding

        for i in range(1, 16):
            codec_i_embedding = model.talker.code_predictor.get_input_embeddings()[i - 1](codec_ids[:, :, i])
            codec_i_embedding = codec_i_embedding * codec_mask.unsqueeze(-1)
            input_embeddings = input_embeddings + codec_i_embedding

        # Forward pass
        outputs = model.talker(
            inputs_embeds = input_embeddings[:, :-1, :],
            attention_mask = attention_mask[:, :-1],
            labels = codec_0_labels[:, 1:],
            output_hidden_states = True,
        )

        # Sub-talker loss for multi-codebook prediction
        hidden_states = outputs.hidden_states[0][-1]
        talker_hidden_states = hidden_states[codec_mask[:, :-1]]
        talker_codec_ids = codec_ids[codec_mask]
        _, sub_talker_loss = model.talker.forward_sub_talker_finetune(
            talker_codec_ids, talker_hidden_states
        )

        loss = outputs.loss + 0.3 * sub_talker_loss

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        num_steps += 1

        if step % 5 == 0:
            print(f"  Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / max(num_steps, 1)
    print(f"Epoch {epoch} complete — Avg Loss: {avg_loss:.4f}")

    # Save checkpoint
    output_dir = os.path.join(OUTPUT_DIR, f"checkpoint-epoch-{epoch}")
    shutil.copytree(local_model_path, output_dir, dirs_exist_ok = True)

    # Update config with speaker info
    config_file = os.path.join(output_dir, "config.json")
    with open(config_file, "r") as f:
        config_dict = json.load(f)
    config_dict["tts_model_type"] = "custom_voice"
    config_dict.setdefault("talker_config", {})["spk_id"] = {SPEAKER_NAME: 3000}
    config_dict["talker_config"]["spk_is_dialect"] = {SPEAKER_NAME: False}
    with open(config_file, "w") as f:
        json.dump(config_dict, f, indent = 2, ensure_ascii = False)

    # Save model weights with speaker embedding
    from safetensors.torch import save_file
    state_dict = {k: v.detach().cpu() for k, v in model.state_dict().items()
                  if not k.startswith("speaker_encoder")}
    weight = state_dict["talker.model.codec_embedding.weight"]
    state_dict["talker.model.codec_embedding.weight"][3000] = (
        target_speaker_embedding[0].detach().to(weight.device).to(weight.dtype)
    )
    save_file(state_dict, os.path.join(output_dir, "model.safetensors"))
    print(f"  Checkpoint saved to {output_dir}")

print("\nTraining complete!")

In [ ]:
# @title Show memory stats
gpu_stats = torch.cuda.get_device_properties(0)
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory % of max memory = {round(used_memory / max_memory * 100, 3)} %.")

<a name="Inference"></a>
### Inference
Let's generate speech with our fine-tuned model! The model should have learned the voice characteristics from the training data.

In [ ]:
#@title Generate Speech

import soundfile as sf
from qwen_tts import Qwen3TTSModel

# Load the fine-tuned model
CHECKPOINT = "output/checkpoint-epoch-2"
SPEAKER_NAME = "elise"

finetuned_tts = Qwen3TTSModel.from_pretrained(
    CHECKPOINT,
    device_map = "cuda:0",
    dtype = torch.bfloat16,
)

# Generate speech
test_texts = [
    "Hello! My name is Elise and I am a speech synthesis model fine-tuned with Unsloth.",
    "The quick brown fox jumps over the lazy dog.",
    "Welcome to the world of text to speech fine-tuning!",
]

for i, text in enumerate(test_texts):
    print(f"Generating: '{text}'")
    wavs, sr = finetuned_tts.generate_custom_voice(
        text = text,
        speaker = SPEAKER_NAME,
    )

    output_file = f"generated_{i}.wav"
    sf.write(output_file, wavs[0], sr)
    print(f"  Saved to {output_file}")

    # Play in notebook
    from IPython.display import Audio, display
    display(Audio(wavs[0], rate = sr))

print("\nInference complete!")

<a name="Save"></a>
### Saving, loading finetuned models
The fine-tuned model checkpoints are already saved during training. You can upload them to Hugging Face Hub for sharing.

In [ ]:
# Upload to Hugging Face Hub
if False:
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_folder(
        folder_path = "output/checkpoint-epoch-2",
        repo_id = "YOUR_USERNAME/qwen3-tts-elise-finetuned",
        repo_type = "model",
        token = "YOUR_HF_TOKEN",
    )

# Or save locally and download
# The checkpoint at output/checkpoint-epoch-2 contains:
# - model.safetensors (model weights with speaker embedding)
# - config.json (updated with speaker info)
# - All other model files from the base model

In [ ]:
# To load a saved fine-tuned model later:
if False:
    from qwen_tts import Qwen3TTSModel
    tts = Qwen3TTSModel.from_pretrained(
        "output/checkpoint-epoch-2",  # or your HF repo ID
        device_map = "cuda:0",
        dtype = torch.bfloat16,
    )
    wavs, sr = tts.generate_custom_voice(
        text = "Hello world!",
        speaker = "elise",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>